# TF-IDF & BM25: Keyword-Based Retrieval Demo

This notebook demonstrates two classic **keyword-based (lexical) retrieval** techniques used in search engines and information retrieval systems, before diving into modern semantic/embedding search:

1. **TF-IDF (Term Frequency–Inverse Document Frequency)** — scores documents based on how important a term is to a document relative to a corpus.
2. **BM25 (Best Matching 25)** — a probabilistic ranking function that improves on TF-IDF with term-frequency saturation and document-length normalization. It is the default ranking algorithm in Elasticsearch/Lucene.

We will:
- Build a small text corpus.
- Implement TF-IDF from scratch (and with `scikit-learn`).
- Implement BM25 from scratch (and with the `rank_bm25` library).
- Run queries against the corpus and compare rankings side-by-side.
- Discuss strengths/weaknesses of each approach.


## 1. Setup

Install/import the libraries we need. `scikit-learn` gives us `TfidfVectorizer`; `rank_bm25` gives us a ready-made BM25 implementation to check our from-scratch version against.

In [1]:
# If running locally and these aren't installed yet, uncomment:
# !pip install scikit-learn rank_bm25 numpy pandas -q

import math
import re
import string
from collections import Counter

import numpy as np
import pandas as pd


## 2. Sample corpus

A tiny corpus of documents about programming languages / data topics — small enough to inspect by eye, varied enough to show ranking differences.

In [2]:
corpus = [
    "Python is a popular programming language for data science and machine learning.",
    "Java is a widely used object oriented programming language for enterprise applications.",
    "Machine learning models require large amounts of data for training.",
    "Deep learning is a subset of machine learning based on neural networks.",
    "Python has simple syntax which makes it a great language for beginners.",
    "Data science combines statistics, programming, and domain knowledge to extract insights.",
    "Neural networks are inspired by the structure of the human brain.",
    "JavaScript is the primary programming language used for web development.",
    "Natural language processing enables computers to understand human language.",
    "Big data technologies help process and analyze massive datasets efficiently.",
]

df = pd.DataFrame({"doc_id": range(len(corpus)), "text": corpus})
df


,doc_id,text
0,0,Python is a popular programming language for d...
1,1,Java is a widely used object oriented programm...
2,2,Machine learning models require large amounts ...
3,3,Deep learning is a subset of machine learning ...
4,4,Python has simple syntax which makes it a grea...
5,5,"Data science combines statistics, programming,..."
6,6,Neural networks are inspired by the structure ...
7,7,JavaScript is the primary programming language...
8,8,Natural language processing enables computers ...
9,9,Big data technologies help process and analyze...


## 3. Text preprocessing

Both TF-IDF and BM25 operate on **tokens**, so we lowercase, strip punctuation, and split on whitespace. (No stemming/lemmatization here to keep things transparent — feel free to extend it.)

In [3]:
def tokenize(text: str):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    tokens = text.split()
    return tokens

tokenized_corpus = [tokenize(doc) for doc in corpus]
for i, toks in enumerate(tokenized_corpus):
    print(i, toks)


0 ['python', 'is', 'a', 'popular', 'programming', 'language', 'for', 'data', 'science', 'and', 'machine', 'learning']
1 ['java', 'is', 'a', 'widely', 'used', 'object', 'oriented', 'programming', 'language', 'for', 'enterprise', 'applications']
2 ['machine', 'learning', 'models', 'require', 'large', 'amounts', 'of', 'data', 'for', 'training']
3 ['deep', 'learning', 'is', 'a', 'subset', 'of', 'machine', 'learning', 'based', 'on', 'neural', 'networks']
4 ['python', 'has', 'simple', 'syntax', 'which', 'makes', 'it', 'a', 'great', 'language', 'for', 'beginners']
5 ['data', 'science', 'combines', 'statistics', 'programming', 'and', 'domain', 'knowledge', 'to', 'extract', 'insights']
6 ['neural', 'networks', 'are', 'inspired', 'by', 'the', 'structure', 'of', 'the', 'human', 'brain']
7 ['javascript', 'is', 'the', 'primary', 'programming', 'language', 'used', 'for', 'web', 'development']
8 ['natural', 'language', 'processing', 'enables', 'computers', 'to', 'understand', 'human', 'language']
9 [

## 4. TF-IDF from scratch

**Term Frequency (TF)**: how often a term appears in a document (often normalized by document length).

$$TF(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total terms in } d}$$

**Inverse Document Frequency (IDF)**: how rare a term is across the whole corpus — rare terms are more informative.

$$IDF(t) = \ln\left(\frac{N}{1 + DF(t)}\right) + 1$$

where $N$ is the number of documents and $DF(t)$ is the number of documents containing term $t$.

**TF-IDF score**: $TFIDF(t, d) = TF(t, d) \times IDF(t)$

A document's relevance to a query is then the **sum of TF-IDF scores** for the query terms it contains (or the cosine similarity between TF-IDF vectors — we show both).

In [4]:
class TFIDFFromScratch:
    def __init__(self, tokenized_docs):
        self.docs = tokenized_docs
        self.N = len(tokenized_docs)
        self.vocab = sorted(set(t for doc in tokenized_docs for t in doc))
        self.doc_freq = self._compute_doc_freq()
        self.idf = self._compute_idf()
        self.tfidf_matrix = self._compute_tfidf_matrix()

    def _compute_doc_freq(self):
        df = Counter()
        for doc in self.docs:
            for term in set(doc):
                df[term] += 1
        return df

    def _compute_idf(self):
        idf = {}
        for term in self.vocab:
            idf[term] = math.log(self.N / (1 + self.doc_freq[term])) + 1
        return idf

    def _tf(self, doc):
        counts = Counter(doc)
        total = len(doc)
        return {term: c / total for term, c in counts.items()}

    def _compute_tfidf_matrix(self):
        matrix = []
        for doc in self.docs:
            tf = self._tf(doc)
            vec = {term: tf.get(term, 0.0) * self.idf[term] for term in tf}
            matrix.append(vec)
        return matrix

    def score(self, query_tokens):
        """Sum of TF-IDF weights of query terms present in each document (simple lexical scoring)."""
        scores = []
        for vec in self.tfidf_matrix:
            s = sum(vec.get(term, 0.0) for term in query_tokens)
            scores.append(s)
        return np.array(scores)

    def cosine_score(self, query_tokens):
        """Cosine similarity between the query's TF-IDF vector and each document's TF-IDF vector."""
        q_tf = Counter(query_tokens)
        q_total = len(query_tokens)
        q_vec = {t: (c / q_total) * self.idf.get(t, 0.0) for t, c in q_tf.items()}

        def cos_sim(v1, v2):
            common = set(v1) & set(v2)
            num = sum(v1[t] * v2[t] for t in common)
            norm1 = math.sqrt(sum(v ** 2 for v in v1.values()))
            norm2 = math.sqrt(sum(v ** 2 for v in v2.values()))
            if norm1 == 0 or norm2 == 0:
                return 0.0
            return num / (norm1 * norm2)

        return np.array([cos_sim(q_vec, doc_vec) for doc_vec in self.tfidf_matrix])


tfidf_model = TFIDFFromScratch(tokenized_corpus)
print("Vocabulary size:", len(tfidf_model.vocab))
print("Sample IDF values:")
for term in ["python", "language", "learning", "data", "javascript"]:
    print(f"  {term:12s} -> {tfidf_model.idf.get(term, 0):.3f}")


Vocabulary size: 70
Sample IDF values:
  python       -> 2.204
  language     -> 1.511
  learning     -> 1.916
  data         -> 1.693
  javascript   -> 2.609


### 4.1 Query the from-scratch TF-IDF model

In [5]:
def show_ranking(scores, top_k=5, label="Score"):
    ranked = np.argsort(-scores)[:top_k]
    for rank, idx in enumerate(ranked, start=1):
        print(f"{rank}. [doc {idx}] {label}={scores[idx]:.4f}  |  {corpus[idx]}")

query = "python programming language"
query_tokens = tokenize(query)
print(f"Query: '{query}'  ->  tokens: {query_tokens}\n")

print("== TF-IDF (sum of term weights) ==")
scores_sum = tfidf_model.score(query_tokens)
show_ranking(scores_sum, label="TF-IDF sum")

print("\n== TF-IDF (cosine similarity) ==")
scores_cos = tfidf_model.cosine_score(query_tokens)
show_ranking(scores_cos, label="cosine")


Query: 'python programming language'  ->  tokens: ['python', 'programming', 'language']

== TF-IDF (sum of term weights) ==
1. [doc 0] TF-IDF sum=0.4507  |  Python is a popular programming language for data science and machine learning.
2. [doc 8] TF-IDF sum=0.3357  |  Natural language processing enables computers to understand human language.
3. [doc 7] TF-IDF sum=0.3204  |  JavaScript is the primary programming language used for web development.
4. [doc 4] TF-IDF sum=0.3096  |  Python has simple syntax which makes it a great language for beginners.
5. [doc 1] TF-IDF sum=0.2670  |  Java is a widely used object oriented programming language for enterprise applications.

== TF-IDF (cosine similarity) ==
1. [doc 0] cosine=0.4792  |  Python is a popular programming language for data science and machine learning.
2. [doc 4] cosine=0.2762  |  Python has simple syntax which makes it a great language for beginners.
3. [doc 7] cosine=0.2368  |  JavaScript is the primary programming language us

### 4.2 Cross-check with `scikit-learn`'s `TfidfVectorizer`

This validates our from-scratch implementation against a widely-used library.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(corpus)
query_vector = vectorizer.transform([query])

sklearn_scores = cosine_similarity(query_vector, doc_vectors).flatten()
show_ranking(sklearn_scores, label="sklearn cosine")


1. [doc 0] sklearn cosine=0.4971  |  Python is a popular programming language for data science and machine learning.
2. [doc 4] sklearn cosine=0.2848  |  Python has simple syntax which makes it a great language for beginners.
3. [doc 7] sklearn cosine=0.2423  |  JavaScript is the primary programming language used for web development.
4. [doc 1] sklearn cosine=0.2229  |  Java is a widely used object oriented programming language for enterprise applications.
5. [doc 8] sklearn cosine=0.2046  |  Natural language processing enables computers to understand human language.


## 5. BM25 from scratch

BM25 improves on plain TF-IDF in two key ways:

1. **Term frequency saturation** — the boost from repeated terms diminishes (a term appearing 10 times isn't 10x more relevant than appearing once).
2. **Document length normalization** — long documents naturally contain more term occurrences by chance, so BM25 penalizes/adjusts for document length relative to the average.

### Formula

$$\text{score}(D, Q) = \sum_{t \in Q} IDF(t) \cdot \frac{f(t, D) \cdot (k_1 + 1)}{f(t, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

Where:
- $f(t, D)$ = frequency of term $t$ in document $D$
- $|D|$ = length of document $D$ (number of tokens)
- $\text{avgdl}$ = average document length across the corpus
- $k_1$ (typically 1.2–2.0) = controls term-frequency saturation
- $b$ (typically 0.75) = controls how much document-length normalization is applied

BM25's IDF is usually computed as:

$$IDF(t) = \ln\left(\frac{N - DF(t) + 0.5}{DF(t) + 0.5} + 1\right)$$

In [7]:
class BM25FromScratch:
    def __init__(self, tokenized_docs, k1=1.5, b=0.75):
        self.docs = tokenized_docs
        self.N = len(tokenized_docs)
        self.k1 = k1
        self.b = b

        self.doc_lens = [len(doc) for doc in tokenized_docs]
        self.avgdl = sum(self.doc_lens) / self.N

        self.doc_term_freqs = [Counter(doc) for doc in tokenized_docs]

        self.doc_freq = Counter()
        for doc in tokenized_docs:
            for term in set(doc):
                self.doc_freq[term] += 1

        self.idf = self._compute_idf()

    def _compute_idf(self):
        idf = {}
        for term, df in self.doc_freq.items():
            idf[term] = math.log((self.N - df + 0.5) / (df + 0.5) + 1)
        return idf

    def score(self, query_tokens):
        scores = np.zeros(self.N)
        for idx, term_freqs in enumerate(self.doc_term_freqs):
            doc_len = self.doc_lens[idx]
            s = 0.0
            for term in query_tokens:
                if term not in term_freqs:
                    continue
                f = term_freqs[term]
                idf = self.idf.get(term, 0.0)
                numerator = f * (self.k1 + 1)
                denominator = f + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
                s += idf * (numerator / denominator)
            scores[idx] = s
        return scores


bm25_model = BM25FromScratch(tokenized_corpus)
print("Average document length:", round(bm25_model.avgdl, 2))
print("Sample BM25 IDF values:")
for term in ["python", "language", "learning", "data", "javascript"]:
    print(f"  {term:12s} -> {bm25_model.idf.get(term, 0):.3f}")


Average document length: 10.9
Sample BM25 IDF values:
  python       -> 1.482
  language     -> 0.693
  learning     -> 1.145
  data         -> 0.894
  javascript   -> 1.992


### 5.1 Query the from-scratch BM25 model

In [8]:
print(f"Query: '{query}'  ->  tokens: {query_tokens}\n")
bm25_scores = bm25_model.score(query_tokens)
show_ranking(bm25_scores, label="BM25")


Query: 'python programming language'  ->  tokens: ['python', 'programming', 'language']

1. [doc 0] BM25=2.9353  |  Python is a popular programming language for data science and machine learning.
2. [doc 4] BM25=2.0803  |  Python has simple syntax which makes it a great language for beginners.
3. [doc 7] BM25=1.6482  |  JavaScript is the primary programming language used for web development.
4. [doc 1] BM25=1.5180  |  Java is a widely used object oriented programming language for enterprise applications.
5. [doc 8] BM25=1.0490  |  Natural language processing enables computers to understand human language.


### 5.2 Cross-check with the `rank_bm25` library

In [9]:
from rank_bm25 import BM25Okapi

rank_bm25_model = BM25Okapi(tokenized_corpus)  # defaults: k1=1.5, b=0.75
lib_scores = rank_bm25_model.get_scores(query_tokens)
show_ranking(np.array(lib_scores), label="rank_bm25")


1. [doc 0] rank_bm25=1.5224  |  Python is a popular programming language for data science and machine learning.
2. [doc 4] rank_bm25=1.1706  |  Python has simple syntax which makes it a great language for beginners.
3. [doc 7] rank_bm25=0.3819  |  JavaScript is the primary programming language used for web development.
4. [doc 5] rank_bm25=0.3662  |  Data science combines statistics, programming, and domain knowledge to extract insights.
5. [doc 1] rank_bm25=0.3518  |  Java is a widely used object oriented programming language for enterprise applications.


## 6. Side-by-side comparison: TF-IDF vs BM25

Let's put all rankings for the same query into one table to compare directly.

In [10]:
comparison = pd.DataFrame({
    "doc_id": range(len(corpus)),
    "text": corpus,
    "tfidf_cosine": scores_cos,
    "bm25_scratch": bm25_scores,
    "bm25_library": lib_scores,
}).sort_values("bm25_scratch", ascending=False).reset_index(drop=True)

comparison


,doc_id,text,tfidf_cosine,bm25_scratch,bm25_library
0,0,Python is a popular programming language for d...,0.479226,2.935271,1.522365
1,4,Python has simple syntax which makes it a grea...,0.276241,2.080280,1.170614
2,7,JavaScript is the primary programming language...,0.236810,1.648206,0.381915
3,1,Java is a widely used object oriented programm...,0.212142,1.518027,0.351751
4,8,Natural language processing enables computers ...,0.198435,1.048984,0.000000
5,5,"Data science combines statistics, programming,...",0.117018,0.890143,0.366213
6,2,Machine learning models require large amounts ...,0.000000,0.000000,0.000000
7,3,Deep learning is a subset of machine learning ...,0.000000,0.000000,0.000000
8,6,Neural networks are inspired by the structure ...,0.000000,0.000000,0.000000
9,9,Big data technologies help process and analyze...,0.000000,0.000000,0.000000


## 7. Trying a different query — where BM25 and TF-IDF can diverge

BM25's length normalization and term-frequency saturation tend to matter most when documents vary in length or when a term is repeated many times in one document. Let's try a query that highlights term-frequency saturation.

In [11]:
query2 = "machine learning neural networks"
query2_tokens = tokenize(query2)
print(f"Query: '{query2}'  ->  tokens: {query2_tokens}\n")

tfidf_scores_2 = tfidf_model.cosine_score(query2_tokens)
bm25_scores_2 = bm25_model.score(query2_tokens)

result2 = pd.DataFrame({
    "doc_id": range(len(corpus)),
    "text": corpus,
    "tfidf_cosine": tfidf_scores_2,
    "bm25": bm25_scores_2,
})

print("Ranked by TF-IDF cosine:")
display(result2.sort_values("tfidf_cosine", ascending=False).reset_index(drop=True))

print("\nRanked by BM25:")
display(result2.sort_values("bm25", ascending=False).reset_index(drop=True))


Query: 'machine learning neural networks'  ->  tokens: ['machine', 'learning', 'neural', 'networks']

Ranked by TF-IDF cosine:


,doc_id,text,tfidf_cosine,bm25
0,3,Deep learning is a subset of machine learning ...,0.623931,5.514380
1,6,Neural networks are inspired by the structure ...,0.277741,2.951026
2,0,Python is a popular programming language for d...,0.269379,2.190775
3,2,Machine learning models require large amounts ...,0.250940,2.378645
4,1,Java is a widely used object oriented programm...,0.000000,0.000000
5,4,Python has simple syntax which makes it a grea...,0.000000,0.000000
6,5,"Data science combines statistics, programming,...",0.000000,0.000000
7,7,JavaScript is the primary programming language...,0.000000,0.000000
8,8,Natural language processing enables computers ...,0.000000,0.000000
9,9,Big data technologies help process and analyze...,0.000000,0.000000



Ranked by BM25:


,doc_id,text,tfidf_cosine,bm25
0,3,Deep learning is a subset of machine learning ...,0.623931,5.514380
1,6,Neural networks are inspired by the structure ...,0.277741,2.951026
2,2,Machine learning models require large amounts ...,0.250940,2.378645
3,0,Python is a popular programming language for d...,0.269379,2.190775
4,1,Java is a widely used object oriented programm...,0.000000,0.000000
5,4,Python has simple syntax which makes it a grea...,0.000000,0.000000
6,5,"Data science combines statistics, programming,...",0.000000,0.000000
7,7,JavaScript is the primary programming language...,0.000000,0.000000
8,8,Natural language processing enables computers ...,0.000000,0.000000
9,9,Big data technologies help process and analyze...,0.000000,0.000000


## 8. A simple interactive-style retrieval function

A convenience wrapper that takes any free-text query and returns the top-k matching documents using BM25 (the generally stronger and more widely-used lexical baseline).

In [12]:
def retrieve(query: str, top_k: int = 3, model="bm25"):
    q_tokens = tokenize(query)
    if model == "bm25":
        scores = bm25_model.score(q_tokens)
    elif model == "tfidf":
        scores = tfidf_model.cosine_score(q_tokens)
    else:
        raise ValueError("model must be 'bm25' or 'tfidf'")

    ranked_idx = np.argsort(-scores)[:top_k]
    results = [(corpus[i], float(scores[i])) for i in ranked_idx if scores[i] > 0]
    return results


for q in ["web development", "training data for models", "beginner friendly language"]:
    print(f"Query: {q!r}")
    for text, score in retrieve(q, top_k=3, model="bm25"):
        print(f"   ({score:.3f}) {text}")
    print()


Query: 'web development'
   (4.139) JavaScript is the primary programming language used for web development.

Query: 'training data for models'
   (5.787) Machine learning models require large amounts of data for training.
   (1.518) Python is a popular programming language for data science and machine learning.
   (0.928) Big data technologies help process and analyze massive datasets efficiently.

Query: 'beginner friendly language'
   (1.049) Natural language processing enables computers to understand human language.
   (0.720) JavaScript is the primary programming language used for web development.
   (0.663) Python has simple syntax which makes it a great language for beginners.



## 9. Summary

| Aspect | TF-IDF | BM25 |
|---|---|---|
| Term frequency handling | Linear (roughly proportional) | Saturating (diminishing returns via `k1`) |
| Document length | Not explicitly normalized (cosine similarity indirectly helps) | Explicitly normalized via `b` and `avgdl` |
| Tunable parameters | None (in the basic form) | `k1`, `b` |
| Typical use | Baseline vector-space retrieval, text similarity | Default ranking function in Elasticsearch, Lucene, OpenSearch |
| Strength | Simple, interpretable, good baseline | Empirically outperforms plain TF-IDF on most ranked-retrieval benchmarks |

**Both are purely lexical (keyword-based)** methods — they only match exact tokens (after preprocessing). Neither understands synonyms or semantic meaning (e.g., a query for "car" won't match a document about "automobile" unless you add stemming/synonym expansion). This is precisely the gap that **dense/embedding-based retrieval** (e.g., sentence transformers + vector search) and **hybrid search** (BM25 + embeddings) aim to close — a natural next step beyond this notebook.
